# OOP Basics

Core object-oriented programming concepts in Python: classes, instantiation, `self`, `__init__`, and inheritance/extension — with comparisons to Java where useful.

In [1]:
# TODO: Implement a class `Dataset` that:
# - takes a list of items in its constructor and stores them
# - supports len(dataset) -> number of items
# - supports dataset[i] -> the i-th item


class Dataset:
    def __init__(self, lst):
        self._lst = lst
    
    def __len__(self):
        return len(self._lst)
    
    def __getitem__(self, indx):
        return self._lst[indx]

d = Dataset([1,2,3])
print(d[0])

1


## `__init__` and dunder methods — what's actually "built in"?

Not every class needs an `__init__`. If you don't define one, the class just inherits the default `__init__` from `object`, which does nothing — it creates a bare instance with no attributes set. You only need to write your own `__init__` if you want to set up instance state (attributes) when the object is created.

`__init__`, `__len__`, `__getitem__` are **dunder methods** ("double underscore"), also called magic methods. They are *not* built-in in the sense that Python writes their logic for you — you implement them yourself. What's special is that the Python interpreter has hardcoded hooks: specific syntax or built-in functions automatically look for a specifically-named method on your object and call it.

- `Dataset(...)` → Python calls `__new__` (creates the object), then `__init__` (initializes it).
- `len(d)` → Python looks for `d.__len__()` and calls it.
- `d[i]` → Python looks for `d.__getitem__(i)` and calls it.

This is how a custom class plugs into built-in syntax/functions — often called operator overloading, or implementing a "protocol." The double-underscore naming is a convention Python itself recognizes for a **fixed, specific set of names** tied to specific operations (`__init__`, `__len__`, `__getitem__`, `__add__`, `__eq__`, `__iter__`, `__str__`, `__enter__`/`__exit__`, etc.). Inventing your own, e.g. `__my_thing__`, does nothing special — it's just a regular method with an odd name; nothing calls it automatically.

In [ ]:
# TODO: Implement a class `LabeledDataset` that extends `Dataset` so that:
# - it takes the same list of items, plus a parallel list of labels
# - dataset[i] -> (item, label) instead of just item
# - len(dataset) still works as before

class Dataset:
    def __init__(self, lst):
        self._lst = lst
    
    def __len__(self):
        return len(self._lst)
    
    def __getitem__(self, indx):
        return self._lst[indx]

class LabeledDataset(Dataset):
    '''
    this class is an extension of the Dataset class
    args: item and label where 
    label[i] is the label for item[i]
    this dataset supports indexing and return the length of the dataset
    '''
    def __init__(self, item, label):
        assert len(item) == len(label), "two arrays should have the same length!!"
        super().__init__(item)
        self._label = label
        
    def __getitem__(self, indx):
        return (self._lst[indx], self._label[indx])
    
    '''
    ## another solution 
    def __getitem__(self, indx):
        return(super().__getitem__(indx), self._label[indx])
    '''

label = [1, [], 2, 4]
item = ['a', 'b', 'c', 'd']

data = LabeledDataset(item, label)
print(len(data))
print(data[1])

4
('b', [])


## Extending a class — lessons from `LabeledDataset`

- **`super()`** is a built-in function (not a dunder — there's no `__super__`) that gives you a proxy to call the parent class's methods.
- **One instance, built in two steps.** Creating a `LabeledDataset` makes a single object. The subclass's own `__init__` sets up the new state (labels); `super().__init__(...)` calls the *parent's* `__init__` on that same `self` to set up the inherited state (`self._lst`). Not two separate objects — one object, initialized in two stages.
- **Overriding doesn't erase the original.** Redefining a method in a subclass doesn't touch or delete the parent's version — it's still intact on the parent class. Python's method lookup checks the instance's actual class first, so the subclass's version is found and used first; the parent's version is still reachable via `super()`.
- **Validate before assigning.** Put input validation (e.g. `assert len(item) == len(label)`) as the *first* line of `__init__`, before any attribute assignment — including before `super().__init__(...)`. If validation fails, the object should never end up half-constructed.
- **Reuse via `super()` instead of duplicating logic.** If a subclass method extends behavior the parent already implements, prefer calling `super().method(...)` over re-implementing it directly:
  ```python
  def __getitem__(self, indx):
      return super().__getitem__(indx), self._label[indx]   # reuses Dataset's lookup
  ```
  rather than reaching into `self._lst[indx]` directly — keeps the logic in one place, so improvements to the parent automatically propagate.
- **Only override what actually changes.** `__len__` didn't need to be redefined in `LabeledDataset` — the inherited version from `Dataset` already does the right thing, since `self._lst` is set up identically via `super().__init__()`.

## `@property` — controlled access to an attribute

A **method** is an action, called explicitly with parentheses: `obj.method()`. A **property** is a method that's *accessed like a plain attribute* — no parentheses — via the `@property` decorator:

```python
class Circle:
    def __init__(self, radius):
        self._radius = radius        # store under a "private" name

    @property
    def radius(self):
        return self._radius          # still just c.radius, no ()
```

Why bother, if it just returns a stored value? Because it gives you a place to **intercept every read/write** and run logic there, without changing how callers use the class:

```python
    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("radius can't be negative")
        self._radius = value
```

`c.radius = -10` now raises `ValueError` — but `c.radius` (read) and `c.radius = 5` (write) look exactly the same to any code using the class as they would with a plain attribute. Nobody has to switch from `c.radius` to `c.get_radius()`/`c.set_radius(5)`; you can even start with a plain public attribute and convert it to a property later without breaking any calling code.

**Common uses beyond validation:**
- **Computed/derived values** — e.g. an `area` property calculated from `self._radius`, with no stored value of its own.
- **Read-only attributes** — define only the getter (no `@x.setter`) and external code can read `c.radius` but assigning `c.radius = 10` raises `AttributeError`.
- **Side effects on write** — e.g. setting one attribute invalidates a cached/derived value elsewhere.
- **Lazy computation/caching** — compute something expensive only the first time it's read, then cache it.

### `@property` is where eager vs. lazy shows up for classes

Eager vs. lazy (see `llms/dataloaders.ipynb`) isn't just about iterators — it applies to attributes too:

- **Eager** — `self.area = 3.14159 * radius ** 2` set directly in `__init__`. Computed immediately at construction, whether or not `area` is ever actually read.
- **Lazy, recomputed every time** — an `area` `@property` that computes from `self._radius` on every read. Deferred until asked, but never cached — if read 1000 times, it recomputes 1000 times.
- **Lazy + cached** — `functools.cached_property` instead of `@property`: computed once, on first access, then the result is stored and reused for every subsequent read. Deferred *and* memoized — the closest attribute-level equivalent to how a generator only computes a value once, when it's actually asked for.

In [ ]:
class Circles:
    def __init__(self, radius):
        self._radius = radius
    
    @property
    def radius(self):
        return self._radius
    

c = Circles(2)
#print(c.radius)
#c.radius = 10 -> cant set it w/o a setter

class Circles:
    def __init__(self, radius):
        self._radius = radius
    
    @property
    def radius(self):
        return self._radius
    
    @radius.setter
    def radius(self, radius):
        if radius <=0:
            raise ValueError("no negative radius!!")
        self._radius = radius

c = Circles(2)
print(c.radius)
c.radius = 10 
print(c.radius)
#c.radius = -1
#print(c.radius)

#### eger implementation of area:
import math
class Circles:
    def __init__(self, radius):
        self.radius = radius # goes through the setter below, which sets _radius and _area
    
    @property
    def radius(self):
        return self._radius
    
    @radius.setter
    def radius(self, radius):
        if radius <=0:
            raise ValueError("no negative radius!!")
        self._radius = radius
        self._area = math.pi * self._radius ** 2   # recomputed eagerly, right when radius changes
        
    @property
    def area(self):
        return self._area   # just returns the cached value, no computation here

c = Circles(2)
print(c.radius)
c.radius = 10 
print(c.radius)
print(c.area)
#c.area = 2
#print(c.area) # gives error!!!

#### naive lazy implementation of area:
import math
class Circles:
    def __init__(self, radius):
        self.radius = radius # we change this to radius instead of _radius
    
    @property
    def radius(self):
        return self._radius
    
    @radius.setter
    def radius(self, radius):
        if radius <=0:
            raise ValueError("no negative radius!!")
        self._radius = radius
       
    @property
    def area(self):
        return math.pi * self._radius ** 2 
        
c = Circles(2)
print(c.radius)
c.radius = 10 
print(c.radius)
print(c.area)
#c.area = 2
#print(c.area) # gives error!!!

#### better lazy implementation of area:
import math
class Circles:
    def __init__(self, radius):
        self.radius = radius # we change this to radius instead of _radius
        self._area = None
    
    @property
    def radius(self):
        return self._radius
    
    @radius.setter
    def radius(self, radius):
        if radius <=0:
            raise ValueError("no negative radius!!")
        self._radius = radius
        self._area = None
       
    @property
    def area(self):
        self._area = math.pi * self._radius ** 2 if self._area is None else self._area
        return self._area

c = Circles(2)
print(c.radius)
c.radius = 10 
print(c.radius)
print(c.area)
print(c.area)
#c.area = 2
#print(c.area) # gives error!!!


2
10
2
10
314.1592653589793
2
10
314.1592653589793
2
10
314.1592653589793
314.1592653589793


## Context managers — `__enter__` / `__exit__`

A **context manager** is any object that hooks into the `with` statement via two more dunder methods:

```python
class Foo:
    def __enter__(self):
        ...            # setup — runs when the `with` block is entered
        return self      # whatever this returns becomes the `as x` value

    def __exit__(self, exc_type, exc_val, exc_tb):
        ...            # cleanup — runs when the block exits, even on an exception
```

```python
with Foo() as x:
    do_stuff(x)
```

- `with Foo()` → Python calls `Foo().__enter__()`.
- Whatever `__enter__` **returns** is what `as x` binds to — it doesn't have to be `self`. It could be a different object entirely (`open()`'s `__enter__` returns the file object itself, which is why `as f` gives you the file), or `None`/nothing if there's nothing useful to hand back (`torch.no_grad()` is rarely used with `as`, since the point is the side effect — disabling gradient tracking — not a returned value).
- `__exit__` always runs on the way out, whether the block finished normally or raised — that guarantee (cleanup always happens) is the entire point, same idea as `finally`.
- `__exit__`'s three args describe an exception if one occurred inside the block (all `None` if it didn't): `exc_type` is the exception class (e.g. `ValueError`), `exc_val` is the exception instance itself (has the actual message/args), and `exc_tb` is the traceback (where in the code it happened). `__exit__` doesn't *catch* the exception in the try/except sense — it can't stop the exception from happening, it's just notified after the fact, once the exception already exists, and gets a chance to react to it. If `__exit__` returns a **truthy** value, the exception is swallowed there — the `with` block, and the caller, act like nothing happened. Returning `None`/`False` (the default) lets it propagate normally, same as if `__exit__` weren't involved at all.

This is exactly the pattern behind `torch.no_grad()`, `open()`, `unittest.mock.patch()`.

In [13]:
# TODO 1: Implement a class-based context manager `Timer` that:
# - records the start time in __enter__ (see the `time` module: time.time() or time.perf_counter())
# - records elapsed time in __exit__, stored as self.elapsed
# - used like:


import time
import numpy as np

class Timer:
    def __enter__(self):
        self.elapsed = time.time()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.time() - self.elapsed
        return self.elapsed


with Timer() as t:
    print("I am so happy lalalala!")
    a = np.array(5)
print(t.elapsed)

# TODO 2 (bonus): implement the same thing again, but using @contextlib.contextmanager
# instead of a class -- a generator function with exactly one `yield` in it, where
# code before the yield is __enter__ and code after the yield is __exit__.
# Compare: which style do you find more readable for this case?


I am so happy lalalala!
0.0


In [ ]:
# TODO: Implement a class `Vector` that:
# - __init__(self, values) -- stores a list of numbers
# - a @staticmethod `dot(v1, v2)` that returns the dot product of two Vector instances
#   (pure function -- doesn't need self or cls, just the two arguments passed in)
# - a @classmethod `zeros(cls, dim)` that returns a new Vector of `dim` zeros
#   (an alternate constructor -- needs `cls` to build the right type, no existing instance)
#
# Then verify all of these work:
# v1 = Vector([1, 2, 3])
# v2 = Vector([4, 5, 6])
# print(Vector.dot(v1, v2))     # called on the class
# print(v1.dot(v1, v2))          # also works via an instance -- self still isn't passed
# z = Vector.zeros(3)
# print(z.values)

class Vector:
    def __init__(self, values):
        self._values = values
    
    @property
    def values(self):
        return self._values

    def __len__(self):
        return len(self.values)
    
    def __getitem__(self, i):
        return self.values[i]
    
    @staticmethod
    def dot(V1, V2):
        assert len(V1) == len(V2), "arrays should be of the same length"
        a = 0.
        for i in range(len(V1)):
            a += V1[i] * V2[i]
        return a
    
    @classmethod
    def zeros(cls, dim):
        return cls([0] * dim)

v1 = Vector([1, 2, 3])
v2 = Vector([4, 5, 6])
print(Vector.dot(v1, v2))   
z = Vector.zeros(3)
print(len(z), z.values)

32.0
3 [0, 0, 0]


## `@staticmethod` — a function that just happens to live in a class

A **static method** doesn't receive `self` or `cls` — it's a regular function, namespaced inside the class purely for organization, not because it needs any instance or class data.

```python
class MathUtils:
    @staticmethod
    def square(x):
        return x * x

MathUtils.square(5)   # 25 -- called on the class, no instance needed
m = MathUtils()
m.square(5)             # 25 -- also callable on an instance, still no self passed in
```

The three method "flavors," contrasted:
- **Instance method** (no decorator, `self` first) — needs a specific object's state to do its job.
- **`@classmethod`** (`cls` first, not an instance) — operates on the class itself; the common use is an alternate constructor (e.g. `Dataset.from_csv(path)` that builds and returns a new instance).
- **`@staticmethod`** — needs neither `self` nor `cls`. Grouped in the class because it's conceptually related, not because it depends on any instance/class data.

Rule of thumb: if a method body never references `self` or `cls`, it should probably be `@staticmethod` — that's usually the tell, whether writing your own or reading someone else's.